# Machine Learning on Microcontrollers: IMX Exercise

## Dependency Installation
- Install the required dependencies for the whole Jupyter Notebook

In [ ]:
# Cell 1
# This cell installs the required dependencies in Google Colab.
# Openjdk is used for the imxconv-pt command from the imx500-converter[pt]

!sudo apt-get update
!apt-get install openjdk-25-jdk-headless
!pip install model_compression_toolkit onnx onnxscript imx500-converter[pt] thop

## Setup Jupyter Notebook

- Import dependencies
- Make results reproducable
- Select device

In [ ]:
# Cell 2

# The following environment variable is used to make CUDA results reproducable
# https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

# Import dependencies.
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import model_compression_toolkit as mct
import matplotlib.pyplot as plt
import numpy as np
import random
import thop
from torchvision import transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from google.colab import files
from tqdm import tqdm

# Make results reproducable by setting the seeds and setting PyTorch to use
# deterministic algorithms.
# For our application, only the settings of PyTorch would be required.
# The others settings are added so there is no need to worry when expanding the code.
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

torch.use_deterministic_algorithms(True)

# Select cuda (GPU) device if available, otherwise run on cpu.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Create Model

- Define the model in a structured way


In [ ]:
# Cell 3
# Defines our model and tests if it outputs the expected dimensions.

# Basic residual block
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            # Projection shortcut to match dimensions
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


# ResNet like model
class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()

        in_channels = 3
        planes = [16, 32, 64]
        blocks = [4, 4, 4]

        # Initial convolution
        self.initial_conv = nn.Conv2d(in_channels, planes[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.initial_bn = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Groups
        self.in_planes = planes[0]
        self.layer1 = self._make_layer(planes[0], blocks[0], stride=1)
        self.layer2 = self._make_layer(planes[1], blocks[1], stride=planes[1] // planes[0])
        self.layer3 = self._make_layer(planes[2], blocks[2], stride=planes[2] // planes[1])

        # Global average pooling + fully connected network
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

        # Weight initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # TODO1 Add Softmax
        # Instanciate a softmax layer.
        self.sm = nn.Softmax(dim=-1)
        # END TODO1

    def _make_layer(self, planes, blocks, stride):
        layers = []

        # First block may downsample
        layers.append(BasicBlock(self.in_planes, planes, stride))
        self.in_planes = planes

        # Remaining blocks
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_planes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.relu(self.initial_bn(self.initial_conv(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        # TODO2 Add Softmax
        # Use the instanciated softmax layer from TODO1 to modify the output
        out = self.sm(out)
        # END TODO2
        return out

if __name__ == "__main__":
    model = ResNet(num_classes=10)
    x = torch.randn(1, 3, 64, 64)
    y = model(x)

    # As CIFAR-10 has 10 classes, we expect torch.Size([1, 10])
    assert y.shape == torch.Size([1, 10])
    print(f"Output shape: {y.shape}")

## Test the Model and print Parameter Counts
- Test the models output vector
- Output information about model parameters

In [ ]:
# Cell 4
if __name__ == "__main__":
    model = ResNet(num_classes=10)
    x = torch.randn(1, 3, 64, 64)
    y = model(x)

    # Print parameters per named layer
    for name, param in model.named_parameters():
      print(f"{name} : {param.numel()}")

    # Print total number of parameters of the model
    print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

    macs, params = thop.profile(model, inputs=(x, ))
    macs, params = thop.clever_format([macs, params], "%.3f")
    print(f"\nMACs: {macs}, Params:{params}\n")


## ONNX Export
- Export ONNX of untrained floating point model

In [ ]:
# Cell 5
# Exports the model in ONNX format.

model_for_export = ResNet()
model_for_export.eval()
example_inputs = (torch.randn(1, 3, 64, 64),)

torch.onnx.export(
    model_for_export,
    example_inputs,
    "floating_point_model_untrained.onnx",
    export_params=True,
    opset_version=18,
    input_names=["input"],
    output_names=["output"],
    dynamo=True,
    do_constant_folding=False,
)

files.download("floating_point_model_untrained.onnx")

## Data Transformations
- Add data transformations to adapt data to requirements and augment the training data

In [ ]:
# Cell 6
# Define the batch size and the data transformations

data_transforms = {
    "train": transforms.Compose([
        # TODO3 Add training data transformations
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(5, translate=(0, 0.05), scale=(0.95, 1.05), shear=10),
        # END TODO3
        transforms.ToTensor(),
    ]),
    "test": transforms.Compose([
        # TODO4 Add test data transformations
        transforms.Resize((64, 64)),
        # END TODO4
        transforms.ToTensor(),
    ]),
}

## Data Import
- Import the training and test data

In [ ]:
# Cell 7

batch_size = 16

train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=data_transforms["train"])

train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)

test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=data_transforms["test"])

test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

## Class Definitions
- Define the number of classes and the class labels

In [ ]:
# Cell 8

num_classes = 10
class_labels = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]


Visualize the Data
- Visualize the data for better understanding

In [ ]:
# Cell 9

# Collect 4 images per class
images_per_class = {cls: [] for cls in range(10)}
needed = 4
for imgs, labels in test_loader:
    for img, label in zip(imgs, labels):
        if len(images_per_class[label.item()]) < needed:
            images_per_class[label.item()].append(img)
    # Stop early if all collected
    if all(len(v) == needed for v in images_per_class.values()):
        break

# Create the grid
fig1, axes1 = plt.subplots(needed, 10, figsize=(20, 8))

for class_idx in range(10):
    for row in range(needed):
        img = images_per_class[class_idx][row]
        img = img.permute(1, 2, 0)  # CHW → HWC

        axes1[row, class_idx].imshow(img)
        axes1[row, class_idx].axis("off")

        # Add class name at top row
        if row == 0:
            axes1[row, class_idx].set_title(class_labels[class_idx])

fig1.suptitle("Test Data")
plt.tight_layout()
plt.show()

# Collect 4 images per class
images_per_class = {cls: [] for cls in range(10)}
needed = 4
for imgs, labels in train_loader:
    for img, label in zip(imgs, labels):
        if len(images_per_class[label.item()]) < needed:
            images_per_class[label.item()].append(img)
    # Stop early if all collected
    if all(len(v) == needed for v in images_per_class.values()):
        break

# Create the grid
fig2, axes2 = plt.subplots(needed, 10, figsize=(20, 8))

for class_idx in range(10):
    for row in range(needed):
        img = images_per_class[class_idx][row]
        img = img.permute(1, 2, 0)  # CHW → HWC

        axes2[row, class_idx].imshow(img)
        axes2[row, class_idx].axis("off")

        # Add class name at top row
        if row == 0:
            axes2[row, class_idx].set_title(class_labels[class_idx])

fig2.suptitle("Train Data")
plt.tight_layout()
plt.show()

## Training Preparation
- Instanciate the model, criterion and loss

In [ ]:
# Cell 10
# Instantiates the model and prepares training

# Create an instance of the model
model = ResNet(num_classes=num_classes).to(device)

# As criterion, CrossEntropyLoss is used
criterion = nn.CrossEntropyLoss()

# As optimizer, Adam is used
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# The number of epochs to train for
epochs = 10


## Training
- Train the model

In [ ]:
# Cell 11

def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch} [Training]", leave=False):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_loss = running_loss / total
    acc = 100. * correct / total
    print(f"Epoch [{epoch}] Train Loss: {train_loss:.4f} | Acc: {acc:.2f}%")
    return (train_loss, acc)

def test(epoch):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc=f"Epoch {epoch} [Test]", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    test_loss = test_loss / total
    acc = 100. * correct / total
    print(f"Epoch [{epoch}] Test Loss: {test_loss:.4f} | Acc: {acc:.2f}%")
    return (test_loss, acc)

best_test_accuracy = 0

train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

for epoch in range(1, epochs + 1):
    train_loss, train_accuracy = train_one_epoch(epoch)
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    test_loss, test_accuracy = test(epoch)
    test_losses.append(test_loss)
    test_accuracies.append(test_accuracy)

    # Save best model
    if test_accuracy > best_test_accuracy:
        best_test_accuracy = test_accuracy
        torch.save(model.state_dict(), "best_fp_model.pth")
        print(f"Saved new best model with acc {best_test_accuracy:.2f}%")


In [ ]:
# Cell 12
# Reloads the best model from storage

model.load_state_dict(torch.load("best_fp_model.pth", weights_only=True))
model.eval()

## Training Evaluation: Accuracies and Loss
- Plot the accuracy and loss over epochs to understand the training process

In [ ]:
# Cell 13
# Plot accuracies and loss

# Number of epochs
epochs = range(1, len(train_accuracies) + 1)

# Create a figure with two subplots side by side
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracies
ax1.plot(epochs, train_accuracies, 'o-', label='Train Accuracy')
ax1.plot(epochs, test_accuracies, 's-', label='Test Accuracy')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy over Epochs')
ax1.legend()
ax1.grid(True)

# Plot losses
ax2.plot(epochs, train_losses, 'o-', label='Train Loss')
ax2.plot(epochs, test_losses, 's-', label='Test Loss')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Loss')
ax2.set_title('Loss over Epochs')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Training Evaluation: Confusion Matrix
- Plot a confusion matrix to understand which classes are predicted well

In [ ]:
# Cell 14

# Change to use torch as long as possible and in the end convert to numpy and list
# Also then remove numpy import
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.append(preds)
        all_labels.append(labels)

cm = confusion_matrix(
    torch.cat(all_labels).cpu().numpy(),
    torch.cat(all_preds).cpu().numpy()
    )

cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

disp = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=class_labels)
disp.plot(cmap='Blues', xticks_rotation=45, values_format=".2f")
disp.ax_.set_title("Confusion Matrix")

## Model Quantization
- Quantize the model using the model compression toolkit from Sony

In [ ]:
# Cell 15

# Create platform capabilities for IMX500
target_platform_cap = mct.get_target_platform_capabilities('pytorch', 'imx500', target_platform_version='v1')

# Add a quantization config

q_config = mct.core.QuantizationConfig(
    activation_error_method=mct.core.QuantizationErrorMethod.MSE,
    weights_error_method=mct.core.QuantizationErrorMethod.MSE,
    weights_bias_correction=True,
    shift_negative_activation_correction=True,
    z_threshold=16,
)

ptq_config = mct.core.CoreConfig(
    quantization_config=q_config,
)

# Define calibration data for the quantization
n_calibration_batches = 20

def representative_dataset_gen():
    dataloader_iter = iter(train_loader)
    for _ in range(n_calibration_batches):
        yield [next(dataloader_iter)[0]]

# Quantize the model
quantized_model, quantization_info = mct.ptq.pytorch_post_training_quantization(
    model,
    representative_data_gen=representative_dataset_gen,
    core_config=ptq_config,
    target_platform_capabilities=target_platform_cap,
)

## Accuracy of Quantized Model
- Test the accuracy of the quantized model

In [ ]:
# Cell 16

correct = 0
total = 0

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = quantized_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print(f'Accuracy of quantized: {100 * correct / total}')

## Quantized Model ONNX Export
- Export quantized model in ONNX format

In [ ]:
# Cell 17

# Save the quantized model in the onnx format
onnx_file_path = 'quantized_model.onnx'

mct.exporter.pytorch_export_model(
    model=quantized_model,
    save_model_path=onnx_file_path,
    repr_dataset=representative_dataset_gen,
)

files.download("quantized_model.onnx")

## Quantized Model IMX Conversion
- Create a memory report
- Convert the ONNX export with the imxconv-pt functionality

In [ ]:
# Cell 18

!imxconv-pt -i quantized_model.onnx -o memory_report --no-input-persistency --memory-report
!zip -r memory_report.zip memory_report
files.download("memory_report.zip")

!imxconv-pt -i quantized_model.onnx -o imxconv_out --no-input-persistency
!zip -r imxconv_out.zip imxconv_out
files.download("imxconv_out.zip")